# Referee four-bit model evaluation

Runs a four-output checkpoint over the validation split and reports metrics separately for the four `encoded_presence` targets. The targets are the same encoded bits used by the current training adapter. These results measure agreement with those encoded bits; they do not establish reviewed text/number detection quality.

The default split is validation. Keep the held-out test split sealed until model selection and thresholds are frozen. A 0.5 threshold is only a reporting reference. The notebook repeats the current trainer's resize-to-224 and `[0, 1]` input transform for checkpoint compatibility. It does not implement the separately planned shared foreground-preprocessing evaluation.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import timm
import torch
from PIL import Image
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
)
from torchvision.transforms import v2

root_candidates = (Path.cwd(), *Path.cwd().parents)
REPOSITORY_ROOT = next((path for path in root_candidates if (path / 'models/referee/data/src/settings.py').is_file()), None)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError('Could not locate the repository root from the notebook working directory.')
DATA_ROOT = REPOSITORY_ROOT / 'models/referee/data/.local/data/train'
PREPARED_DIR = DATA_ROOT / 'prepared'
SPLIT = 'validation'  # Do not use 'test' until evaluation is frozen.
CHECKPOINT_DIR = REPOSITORY_ROOT / 'models/referee/checkpoints/referee_run/weights'
CHECKPOINTS = sorted(CHECKPOINT_DIR.glob('referee-epoch=*-val_mean_ap=*.ckpt'), key=lambda p: p.stat().st_mtime)
if not CHECKPOINTS:
    raise FileNotFoundError(f'No selected epoch checkpoint under {CHECKPOINT_DIR}')
CHECKPOINT_PATH = CHECKPOINTS[-1]
BIT_NAMES = tuple(f'bit_{index}' for index in range(1, 5))
THRESHOLD = 0.5  # Reference point only; tune on validation data.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Split: {SPLIT}; device: {device}')

In [ ]:
manifest_path = PREPARED_DIR / 'manifests' / f'{SPLIT}.jsonl'
rows = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
if any(len(row.get('encoded_presence', [])) != 4 for row in rows):
    raise ValueError('Every manifest row must have four encoded_presence bits.')
support = pd.DataFrame([
    {
        'target': BIT_NAMES[index],
        'positive': sum(row['encoded_presence'][index] == 1 for row in rows),
        'negative': sum(row['encoded_presence'][index] == 0 for row in rows),
        'rows': len(rows),
    }
    for index in range(4)
])
support

## Load the four-output checkpoint

This checks that the checkpoint head matches the four `encoded_presence` bits before loading its weights.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
state = checkpoint.get('state_dict', checkpoint)
model_state = {key.removeprefix('model.'): value for key, value in state.items() if key.startswith('model.')}
if not model_state:
    raise ValueError('Checkpoint has no Lightning model.* weights.')
head_weights = [value for key, value in model_state.items() if key.endswith('head.weight') or key.endswith('head.fc.weight')]
if not head_weights or head_weights[-1].shape[0] != 4:
    observed = None if not head_weights else head_weights[-1].shape[0]
    raise ValueError(f'Expected a four-output encoded_presence checkpoint; found {observed} outputs.')
model = timm.create_model('vit_tiny_patch16_224.augreg_in21k_ft_in1k', pretrained=False, num_classes=4)
model.load_state_dict(model_state, strict=True)
model.to(device).eval()
model

In [ ]:
transform = v2.Compose([v2.Resize((224, 224)), v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
prediction_rows = []
with torch.inference_mode():
    for row in rows:
        image_path = PREPARED_DIR / row['image_path']
        with Image.open(image_path) as image:
            image_tensor = transform(image.convert('RGB')).unsqueeze(0).to(device)
        logits = model(image_tensor)
        if logits.shape != (1, 4):
            raise ValueError(f'Expected [1, 4] logits, received {tuple(logits.shape)}')
        scores = torch.sigmoid(logits[0]).cpu().tolist()
        result = {'sample_id': row['sample_id']}
        for index, bit_name in enumerate(BIT_NAMES):
            result[f'{bit_name}_score'] = scores[index]
            result[f'{bit_name}_target'] = row['encoded_presence'][index]
        prediction_rows.append(result)
predictions = pd.DataFrame(prediction_rows)
print(f'Produced predictions for {len(predictions)} {SPLIT} rows.')
predictions.head()

In [ ]:
metric_rows = []
for bit_name in BIT_NAMES:
    y_true = predictions[f'{bit_name}_target'].astype(int)
    y_score = predictions[f'{bit_name}_score'].astype(float)
    y_pred = (y_score >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metric_rows.append({
        'target': bit_name, 'support': len(y_true),
        'positive': int(y_true.sum()), 'negative': int((y_true == 0).sum()),
        'average_precision': average_precision_score(y_true, y_score) if y_true.nunique() == 2 else None,
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'confusion_tn_fp_fn_tp': (int(tn), int(fp), int(fn), int(tp)),
    })
metrics = pd.DataFrame(metric_rows)
metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for axis, bit_name in zip(axes.flat, BIT_NAMES):
    y_true = predictions[f'{bit_name}_target'].astype(int)
    if y_true.nunique() < 2:
        axis.text(0.5, 0.5, 'Only one target class in this split', ha='center', va='center')
        axis.set_title(f'{bit_name} precision-recall')
        axis.set_axis_off()
        continue
    precision, recall, _ = precision_recall_curve(y_true, predictions[f'{bit_name}_score'])
    axis.plot(recall, precision)
    axis.set(xlabel='Recall', ylabel='Precision', title=f'{bit_name} precision-recall')
    axis.grid(True, alpha=0.3)
fig.tight_layout()